# Block 1 — RAG zu Fuss

Ein vollständiges RAG in drei Schritten — ohne Framework, jeder Schritt sichtbar:

1. **Ingest** — Dokumente chunken, embedden, in Chroma ablegen
2. **LLM-Verbindung** — einfache Anfrage ans Modell (Sanity-Check)
3. **RAG-Abfrage** — Frage → Retrieval → Antwort

Embeddings *und* Generierung laufen über **einen** OpenAI-kompatiblen Endpoint (Ollama oder ein interner Proxy) — kein lokaler ML-Stack. Eine Stelle ist **deine Aufgabe**: die primitive Chunking-Strategie.

## 0. Setup

Ein Client für Embeddings und Chat. URL/Key/Modelle kommen aus der Umgebung (`.env`); Default ist ein lokales Ollama.

In [ ]:
import os
from pathlib import Path

import chromadb
from openai import OpenAI

DATA = Path.cwd().parent / "data" if Path.cwd().name == "notebooks" else Path("data")
CHROMA_PATH = str(DATA.parent / ".chroma")
CHUNK_SIZE = 800
CHAT_MODEL = os.getenv("LLM_MODEL", "llama3.2")
EMBED_MODEL = os.getenv("EMBED_MODEL", "bge-m3")

client = OpenAI(
    base_url=os.getenv("LLM_BASE_URL", "http://localhost:11434/v1"),
    api_key=os.getenv("LLM_API_KEY", "ollama"),
)

def embed(texts):
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [d.embedding for d in resp.data]

print("Dokumente:", [p.name for p in sorted(DATA.glob("*.md"))])

## Schritt 1 — Ingest

### 1a. Chunking — deine Aufgabe ✍️
Schneide den Text alle `CHUNK_SIZE` Zeichen in Stücke — ohne Rücksicht auf Wort- oder Satzgrenzen.

*Tipp:* `range(0, len(text), CHUNK_SIZE)` und `text[i:i+CHUNK_SIZE]`.

In [ ]:
def chunk(text: str) -> list[str]:
    # TODO: gib eine Liste von Textstücken zurück, je CHUNK_SIZE Zeichen lang.
    return []


# Referenz-Lösung (einkommentieren, falls du vergleichen willst):
# def chunk(text: str) -> list[str]:
#     return [text[i:i + CHUNK_SIZE] for i in range(0, len(text), CHUNK_SIZE)]

sample = (DATA / "vector-databases.md").read_text(encoding="utf-8")
print(f"{len(chunk(sample))} Chunks aus vector-databases.md")

### 1b. Embedden + in Chroma speichern
Embeddings kommen vom Endpoint, gespeichert wird file-based in Chroma — kein Server.

In [ ]:
collection = chromadb.PersistentClient(path=CHROMA_PATH).get_or_create_collection(
    "rag_demo", embedding_function=None, metadata={"hnsw:space": "cosine"}
)

for path in sorted(DATA.glob("*.md")):
    chunks = chunk(path.read_text(encoding="utf-8"))
    collection.upsert(
        ids=[f"{path.name}::{i}" for i in range(len(chunks))],
        embeddings=embed(chunks),
        documents=chunks,
        metadatas=[{"source": path.name} for _ in chunks],
    )

print("Chunks in der DB:", collection.count())

## Schritt 2 — Verbindung zum LLM

Eine einfache Chat-Anfrage als Sanity-Check (das Embedden oben nutzte denselben Endpoint bereits).

In [ ]:
ping = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": "Say hello in one short sentence."}],
)
print(ping.choices[0].message.content)

## Schritt 3 — RAG-Abfrage

Frage embedden → ähnlichste Chunks holen → in den Prompt geben → Antwort.

In [ ]:
question = "What is a vector database?"

hits = collection.query(query_embeddings=embed([question]), n_results=4)
context = "\n\n".join(hits["documents"][0])

answer = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[
        {"role": "system", "content": "Answer the question using ONLY the context. If it is not in the context, say so."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ],
)
print(answer.choices[0].message.content)
print("\nQuellen:", [m["source"] for m in hits["metadatas"][0]])

## Geschafft 🎉

Drei Schritte, ein lauffähiger Knowledge-Worker.

**Ausblick Block 2 (Wahlmodul):** Overlap beim Chunking · Reranking · Query Rewriting · Evaluation.